##### **Import functions**

In [13]:
from pyspark.sql import functions as F
print("Gold notebook started")

StatementMeta(, 8773d42a-98b3-494b-b01e-c76873cf12b6, 15, Finished, Available, Finished, False)

Gold notebook started


##### **Read Silver tables**

In [14]:
customers = spark.read.table("silver_customers")
products = spark.read.table("silver_products")
orders = spark.read.table("silver_orders")

print("Customers:", customers.count())
print("Products:", products.count())
print("Orders:", orders.count())

StatementMeta(, 8773d42a-98b3-494b-b01e-c76873cf12b6, 16, Finished, Available, Finished, False)

Customers: 1000
Products: 976
Orders: 2805


##### **Create gold_fact_sales**

In [15]:
fact_sales = orders.join(
    products.select("product_id", "product_name", "category", "unit_price"),
    on="product_id",
    how="inner"
)

fact_sales = fact_sales.withColumn(
    "line_amount",
    F.round(
        F.col("quantity") *
        F.col("unit_price") *
        (1 - F.col("discount_pct") / 100),
        2
    )
)

fact_sales = fact_sales.withColumn(
    "year_month",
    F.date_format("order_date", "yyyy-MM")
)

gold_fact_sales = fact_sales.select(
    "order_id",
    "year_month",
    "order_date",
    "customer_id",
    "product_id",
    "product_name",
    "category",
    "quantity",
    "unit_price",
    "discount_pct",
    "payment_mode",
    "status",
    "line_amount"
)

gold_fact_sales.write.format("delta").mode("overwrite").saveAsTable(
    "gold_fact_sales"
)

print("Gold fact sales created")

StatementMeta(, 8773d42a-98b3-494b-b01e-c76873cf12b6, 17, Finished, Available, Finished, False)

Gold fact sales created


##### **Create gold_sales_by_month_state**

In [16]:
sales = (
    gold_fact_sales
    .filter(F.col("status").isin("Delivered", "Shipped"))
    .join(
        customers.select("customer_id", "state"),
        on="customer_id",
        how="inner"
    )
)

gold_sales_by_month_state = sales.groupBy(
    "year_month", "state"
).agg(
    F.countDistinct("order_id").alias("orders"),
    F.round(F.sum("line_amount"), 2).alias("revenue")
).orderBy(
    "year_month", "state"
)

gold_sales_by_month_state.write.format("delta").mode("overwrite").saveAsTable(
    "gold_sales_by_month_state"
)

print("Monthly sales by state created")

StatementMeta(, 8773d42a-98b3-494b-b01e-c76873cf12b6, 18, Finished, Available, Finished, False)

Monthly sales by state created


##### **Create gold_top_products**

In [18]:
from pyspark.sql.window import Window

product_sales = (
    gold_fact_sales
    .filter(F.col("status").isin("Delivered", "Shipped"))
    .groupBy("product_id", "product_name", "category")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("line_amount"), 2).alias("revenue")
    )
)

rank_window = Window.orderBy(
    F.col("revenue").desc(),
    F.col("product_id").asc()
)

gold_top_products = (
    product_sales
    .withColumn("rank", F.row_number().over(rank_window))
    .filter(F.col("rank") <= 10)
    .select(
        "rank", "product_id", "product_name",
        "category", "units_sold", "revenue"
    )
)

gold_top_products.write.format("delta").mode("overwrite").saveAsTable(
    "gold_top_products"
)

print("Top 10 products created")

StatementMeta(, 8773d42a-98b3-494b-b01e-c76873cf12b6, 20, Finished, Available, Finished, False)

Top 10 products created


##### **Create gold_customer_summary**

In [19]:
customer_sales = gold_fact_sales.filter(
    F.col("status").isin("Delivered", "Shipped")
)

customer_summary = customer_sales.groupBy("customer_id").agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.round(F.sum("line_amount"), 2).alias("total_revenue"),
    F.min("order_date").alias("first_order_date"),
    F.max("order_date").alias("last_order_date")
)

gold_customer_summary = customer_summary.join(
    customers.select("customer_id", "customer_name", "state"),
    on="customer_id",
    how="inner"
)

gold_customer_summary = gold_customer_summary.withColumn(
    "customer_tier",
    F.when(F.col("total_revenue") >= 100000, "Gold")
    .when(F.col("total_revenue") >= 25000, "Silver")
    .otherwise("Bronze")
)

gold_customer_summary = gold_customer_summary.select(
    "customer_id",
    "customer_name",
    "state",
    "total_orders",
    "total_revenue",
    "first_order_date",
    "last_order_date",
    "customer_tier"
)

gold_customer_summary.write.format("delta").mode("overwrite").saveAsTable(
    "gold_customer_summary"
)

print("Customer summary created")

StatementMeta(, 8773d42a-98b3-494b-b01e-c76873cf12b6, 21, Finished, Available, Finished, False)

Customer summary created


##### **Verify the Gold tables**

In [20]:
print("Gold Fact Sales")
display(spark.read.table("gold_fact_sales"))

print("Gold Sales by Month and State")
display(spark.read.table("gold_sales_by_month_state"))

print("Gold Top Products")
display(spark.read.table("gold_top_products"))

print("Gold Customer Summary")
display(spark.read.table("gold_customer_summary"))

StatementMeta(, 8773d42a-98b3-494b-b01e-c76873cf12b6, 22, Finished, Available, Finished, False)

Gold Fact Sales


SynapseWidget(Synapse.DataFrame, 5ebd43d6-d77e-41a8-b7b4-7f92994cadf0)

Gold Sales by Month and State


SynapseWidget(Synapse.DataFrame, 949faba6-aeea-4595-97e1-9733559a286b)

Gold Top Products


SynapseWidget(Synapse.DataFrame, 91f80212-edd8-4c91-9399-c84de3c1562e)

Gold Customer Summary


SynapseWidget(Synapse.DataFrame, 8eabfa6e-99b9-40a2-b5ca-2046d34d34eb)

In [21]:
df = spark.read.table("ctl_source_config")

df.show()

df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("Files/export/ctl_source_config")

StatementMeta(, 8773d42a-98b3-494b-b01e-c76873cf12b6, 23, Finished, Available, Finished, False)

+---------+-------------+----------------+----------------+-----------+----------+---------+
|source_id|  source_file|    bronze_table|    silver_table| key_column|load_order|is_active|
+---------+-------------+----------------+----------------+-----------+----------+---------+
|        1|customers.csv|bronze_customers|silver_customers|customer_id|         1|        Y|
|        2| products.csv| bronze_products| silver_products| product_id|         2|        Y|
|        3|   orders.csv|   bronze_orders|   silver_orders|   order_id|         3|        Y|
+---------+-------------+----------------+----------------+-----------+----------+---------+



##### **Export the tables as CSV files**

In [22]:
export_tables = [
    "ctl_source_config",
    "ctl_run_log",
    "silver_customers",
    "silver_products",
    "silver_orders",
    "silver_rejects",
    "gold_fact_sales",
    "gold_sales_by_month_state",
    "gold_top_products",
    "gold_customer_summary"
]

for table_name in export_tables:
    df = spark.read.table(table_name)

    output_path = "Files/export/" + table_name

    df.coalesce(1).write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(output_path)

    print("Exported:", table_name)

StatementMeta(, 8773d42a-98b3-494b-b01e-c76873cf12b6, 24, Finished, Available, Finished, False)

Exported: ctl_source_config
Exported: ctl_run_log
Exported: silver_customers
Exported: silver_products
Exported: silver_orders
Exported: silver_rejects
Exported: gold_fact_sales
Exported: gold_sales_by_month_state
Exported: gold_top_products
Exported: gold_customer_summary


##### **Check the row counts**

In [23]:
gold_tables = [
    "gold_fact_sales",
    "gold_sales_by_month_state",
    "gold_top_products",
    "gold_customer_summary"
]

for table_name in gold_tables:
    row_count = spark.read.table(table_name).count()
    print(table_name, ":", row_count, "rows")

StatementMeta(, 8773d42a-98b3-494b-b01e-c76873cf12b6, 25, Finished, Available, Finished, False)

gold_fact_sales : 2805 rows
gold_sales_by_month_state : 60 rows
gold_top_products : 10 rows
gold_customer_summary : 872 rows


##### **Add Gold results to the run log**

In [24]:
run_id = "manual"

gold_log = [
    (run_id, "Gold", "gold_fact_sales",
     spark.read.table("gold_fact_sales").count(), "SUCCESS"),
    (run_id, "Gold", "gold_sales_by_month_state",
     spark.read.table("gold_sales_by_month_state").count(), "SUCCESS"),
    (run_id, "Gold", "gold_top_products",
     spark.read.table("gold_top_products").count(), "SUCCESS"),
    (run_id, "Gold", "gold_customer_summary",
     spark.read.table("gold_customer_summary").count(), "SUCCESS")
]

log_df = spark.createDataFrame(
    gold_log,
    ["run_id", "layer", "table_name", "rows_written", "status"]
)

log_df = log_df.withColumn("log_ts", F.current_timestamp())

log_df.write.format("delta").mode("append").saveAsTable("ctl_run_log")

print("Gold run logged successfully")

StatementMeta(, 8773d42a-98b3-494b-b01e-c76873cf12b6, 26, Finished, Available, Finished, False)

Gold run logged successfully


In [1]:
items = notebookutils.fs.ls("Files/export")

for item in items:
    print(item.name, "Folder:", item.isDir)

StatementMeta(, 478f2d50-7164-4ce6-914d-8e691c2d5f71, 3, Finished, Available, Finished, False)

ctl_run_log Folder: True
ctl_source_config Folder: True
gold_customer_summary Folder: True
gold_fact_sales Folder: True
gold_sales_by_month_state Folder: True
gold_top_products Folder: True
silver_customers Folder: True
silver_orders Folder: True
silver_products Folder: True
silver_rejects Folder: True


In [2]:
for folder in notebookutils.fs.ls("Files/export"):
    if folder.isDir:
        print("\nTable:", folder.name)

        for file in notebookutils.fs.ls("Files/export/" + folder.name):
            print("File:", file.name)

StatementMeta(, 478f2d50-7164-4ce6-914d-8e691c2d5f71, 5, Finished, Available, Finished, False)


Table: ctl_run_log
File: _SUCCESS
File: part-00000-cd9124f8-d3fd-4920-895b-bdc252965758-c000.csv

Table: ctl_source_config
File: _SUCCESS
File: part-00000-55daf630-601e-45a6-a38a-92043adb7315-c000.csv

Table: gold_customer_summary
File: _SUCCESS
File: part-00000-df7d227d-15bc-4fae-9a52-5dee864ac184-c000.csv

Table: gold_fact_sales
File: _SUCCESS
File: part-00000-0cf9fde5-2fff-4166-a4ab-3a8eac169e9b-c000.csv

Table: gold_sales_by_month_state
File: _SUCCESS
File: part-00000-a609cfc8-19d1-4985-ba02-c7394f528205-c000.csv

Table: gold_top_products
File: _SUCCESS
File: part-00000-4f5979df-0b64-4ddc-99e4-28271bc1e559-c000.csv

Table: silver_customers
File: _SUCCESS
File: part-00000-dc862be4-75f5-47d6-b872-3d4140ad2999-c000.csv

Table: silver_orders
File: _SUCCESS
File: part-00000-e9677830-013f-4472-a7e9-1dcd60c08bbe-c000.csv

Table: silver_products
File: _SUCCESS
File: part-00000-be1cd616-1c64-4d87-b828-aa4145321d54-c000.csv

Table: silver_rejects
File: _SUCCESS
File: part-00000-5680cb37-449

In [3]:
import os
import zipfile
import notebookutils

local_zip = "/tmp/Fabric_Assessment_Outputs.zip"

with zipfile.ZipFile(local_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in notebookutils.fs.ls("Files/export"):
        if folder.isDir:
            for file in notebookutils.fs.ls("Files/export/" + folder.name):
                if file.name.endswith(".csv"):
                    source = "Files/export/" + folder.name + "/" + file.name
                    temp_file = "/tmp/" + file.name

                    notebookutils.fs.cp(source, "file:" + temp_file)
                    z.write(temp_file, folder.name + ".csv")
                    os.remove(temp_file)

print("ZIP created:", local_zip)

StatementMeta(, 478f2d50-7164-4ce6-914d-8e691c2d5f71, 6, Finished, Available, Finished, False)

ZIP created: /tmp/Fabric_Assessment_Outputs.zip


In [4]:
notebookutils.fs.cp(
    "file:/tmp/Fabric_Assessment_Outputs.zip",
    "Files/export/Fabric_Assessment_Outputs.zip"
)

StatementMeta(, 478f2d50-7164-4ce6-914d-8e691c2d5f71, 7, Finished, Available, Finished, False)

True

In [5]:
for item in notebookutils.fs.ls("Files/export"):
    print(item.name)

StatementMeta(, 478f2d50-7164-4ce6-914d-8e691c2d5f71, 8, Finished, Available, Finished, False)

Fabric_Assessment_Outputs.zip
ctl_run_log
ctl_source_config
gold_customer_summary
gold_fact_sales
gold_sales_by_month_state
gold_top_products
silver_customers
silver_orders
silver_products
silver_rejects
